# Model Validation Notebook

### Plots of Model FC vs Empirical FC

Correlation between average empirical FC and average model FC for the different brain models.

In [53]:
import numpy as np
from scipy.stats import pearsonr
from hbnm.model.utils import subdiag, fisher_z, linearize_map
import matplotlib.pyplot as plt
from hbnm.io import Data
import os
from hbnm.bnm import Bnm


### Surrogate Correlation w/ Empirical FC

Histogram of the correlations with empirical data of the surrogate maps

(**Still have to ask Yasir if this makes sense**)

In [54]:
# Set up data loader
input_dir = "/home/frank/HBNM/data/"
output_dir = "/home/frank/HBNM/outputs/"
data = Data(input_dir, output_dir)  

# Load HCP SC and Empirical FC Matrices
sc, _, fc_obj = data.load_demirtas_neuron_2019_data()

In [55]:
# Load the fitted parameters for all the maps

# Standardized simulation names that match the rest of the code
simulations = ['dopamine', 'gaba', 'norepinephrine', 'nmda', 'myelin']

# Dictionary to store theta values for each simulation
theta_values = {}

# Latest iterations for each simulation (mapping standardized names to file paths)
latest_iterations = {
    'dopamine': 28,
    'gaba': 33, 
    'norepinephrine': 30,
    'nmda': 41,
    'myelin': 35
}

# Mapping from standardized names to file directory names
file_mapping = {
    'dopamine': 'Dopamine_AVG',
    'gaba': 'GABAa_AVG',
    'norepinephrine': 'Norepinephrine_AVG',
    'nmda': 'NMDA_AVG',
    'myelin': 'Heterogenous'
}

# Load theta values for each simulation
for sim in simulations:
    iteration = latest_iterations[sim]
    file_dir = file_mapping[sim]
    fin = data.load(f'{file_dir}/iteration_{iteration}.hdf5', from_output=True)
    theta_values[sim] = fin['theta'][:]
    fin.close()
    print(f"Loaded theta values for {sim} from iteration {iteration}")

Loaded theta values for dopamine from iteration 28
Loaded theta values for gaba from iteration 33
Loaded theta values for norepinephrine from iteration 30
Loaded theta values for nmda from iteration 41
Loaded theta values for myelin from iteration 35


In [56]:
# Linearize and format all the empirical and surrogate maps so that they are shape (1, N)

# Directory containing the receptor vectors
receptor_dir = "/home/frank/HBNM/data/receptor_vectors"

# Directory containing the surrogate maps
surrogate_dir = "/home/frank/HBNM/data/surrogate_maps"

# Dictionary to store all linearized empirical maps
empirical_maps = {}

# Dictionary to store all linearized surrogate maps
surrogate_maps = {}

print("Loading empirical receptor maps...")
print("-" * 40)

# Get all .npy files in the receptor_vectors directory
receptor_files = [f for f in os.listdir(receptor_dir) if f.endswith('.npy')]

# Load and linearize each empirical map
for file in receptor_files:
    # Extract map name (remove 'y_vector_' prefix and '.npy' suffix)
    map_name = file.replace('y_vector_', '').replace('.npy', '')
    
    # Load the map
    map_data = np.load(os.path.join(receptor_dir, file))
    
    # Ensure it's in (1, N) format
    if map_data.ndim == 1:
        map_data = map_data.reshape(1, -1)
    elif map_data.shape[0] != 1:
        map_data = map_data.reshape(1, -1)
    
    # Linearize the map
    linearized_map = linearize_map(map_data)
    
    # Store in dictionary
    empirical_maps[map_name] = linearized_map
    
    print(f"Loaded and linearized {map_name} map: shape {linearized_map.shape}")

print(f"Available empirical maps: {list(empirical_maps.keys())}")
print()

print("Loading surrogate maps...")
print("-" * 40)

# Get all .npy files in the surrogate_maps directory
surrogate_files = [f for f in os.listdir(surrogate_dir) if f.endswith('.npy')]

# Load and linearize each surrogate map
for file in surrogate_files:
    # Extract map name (remove '_surrogates' suffix and '.npy' suffix)
    map_name = file.replace('_surrogates.npy', '')
    
    # Load the surrogate maps
    surrogate_data = np.load(os.path.join(surrogate_dir, file))
    
    print(f"Loaded {map_name} surrogates: shape {surrogate_data.shape}")
    
    # Linearize each surrogate map
    linearized_surrogates = []
    
    # Handle different possible shapes of surrogate data
    if surrogate_data.ndim == 1:
        # Single surrogate map
        linearized_surrogates.append(linearize_map(surrogate_data))
    elif surrogate_data.ndim == 2:
        # Multiple surrogate maps (each row is a surrogate)
        for i in range(surrogate_data.shape[0]):
            surrogate_map = surrogate_data[i, :]
            linearized_surrogates.append(linearize_map(surrogate_map))
    
    # Stack all linearized surrogates
    linearized_surrogates = np.vstack(linearized_surrogates)
    
    # Store in dictionary
    surrogate_maps[map_name] = linearized_surrogates
    
    print(f"Linearized {map_name} surrogates: final shape {linearized_surrogates.shape}")

print(f"Available surrogate maps: {list(surrogate_maps.keys())}")

# Now you have two dictionaries:
# - empirical_maps: contains the original receptor maps
# - surrogate_maps: contains the surrogate maps for statistical testing

Loading empirical receptor maps...
----------------------------------------
Loaded and linearized myelin map: shape (1, 180)
Loaded and linearized nmda map: shape (1, 180)
Loaded and linearized gaba map: shape (1, 180)
Loaded and linearized norepinephrine map: shape (1, 180)
Loaded and linearized dopamine map: shape (1, 180)
Available empirical maps: ['myelin', 'nmda', 'gaba', 'norepinephrine', 'dopamine']

Loading surrogate maps...
----------------------------------------
Loaded gaba surrogates: shape (500, 180)
Linearized gaba surrogates: final shape (500, 180)
Loaded norepinephrine surrogates: shape (500, 180)
Linearized norepinephrine surrogates: final shape (500, 180)
Loaded nmda surrogates: shape (500, 180)
Linearized nmda surrogates: final shape (500, 180)
Loaded dopamine surrogates: shape (500, 180)
Linearized dopamine surrogates: final shape (500, 180)
Loaded myelin surrogates: shape (500, 180)
Linearized myelin surrogates: final shape (500, 180)
Available surrogate maps: ['ga

In [57]:
empirical_fc = subdiag(fc_obj)

In [ ]:
# Process all combinations of surrogate maps and empirical maps

empirical_fc = subdiag(fc_obj)

# Define which maps need inversion
map_invert_mapping = {
    'myelin': True,      # needs inversion
    'gaba': True,        # needs inversion  
    'nmda': False,       # no inversion
    'dopamine': False,   # no inversion
    'norepinephrine': False  # no inversion
}

# Dictionary to store results for each map type
all_results = {}

print("Processing all combinations of empirical and surrogate maps...")

# Process each empirical map type
for map_name in empirical_maps.keys():
    print(f"\n{'='*50}")
    print(f"Processing {map_name.upper()} empirical map")
    print(f"{'='*50}")
    
    # Check if surrogate maps exist for this map type
    if map_name not in surrogate_maps:
        print(f"Warning: Surrogate maps for {map_name} not found. Skipping {map_name}")
        continue
    
    # Get surrogate maps for this map type
    surrogates = surrogate_maps[map_name]
    n_surrogates = surrogates.shape[0]
    print(f"Loaded {n_surrogates} surrogate maps for {map_name}")
    
    # Get corresponding theta values
    if map_name not in theta_values:
        print(f"Warning: Theta values for {map_name} not found. Skipping {map_name}")
        continue
    
    current_theta = theta_values[map_name]
    print(f"Using theta values from {map_name}")
    
    # Store correlations for this map type
    surrogate_correlations = []
    
    # Track None corr_bold cases
    n_surrogates_processed = 0
    n_surrogates_none = 0
    n_surrogates_errors = 0
    
    # Process each surrogate map
    # for i in range(n_surrogates):
    for i in range(min(500, n_surrogates)):  # Limit to 100 surrogates for testing
        # Get the i-th surrogate map and reshape to (1, 180)
        surrogate_map = surrogates[i].reshape(1, -1)
        
        # Get the appropriate invert flag for this map type
        invert_flag = map_invert_mapping.get(map_name, True)  # Default to True if map not found
        
        # Create heterogeneous model with this surrogate
        heterogeneous_shuffled = Bnm(sc, maps=surrogate_map, map_invert_flags=[invert_flag])
        heterogeneous_shuffled.set('w_EI', (current_theta[0,0], current_theta[1,0]))
        heterogeneous_shuffled.set('w_EE', (current_theta[2,0], current_theta[3,0]))
        heterogeneous_shuffled.set('G', current_theta[4,0])
        
        n_surrogates_processed += 1
        
        try:
            heterogeneous_shuffled.moments_method()
            
            # Check if corr_bold exists and is not None
            model_FC_surrogate = heterogeneous_shuffled.get('corr_bold')
            if model_FC_surrogate is None:
                print(f"Warning: corr_bold is None for surrogate {i+1}")
                n_surrogates_none += 1
                continue
                
            model_fc_surrogate = subdiag(model_FC_surrogate)
            
            # Calculate correlation with empirical FC
            corr_surrogate, _ = pearsonr(empirical_fc, model_fc_surrogate)
            surrogate_correlations.append(corr_surrogate)
            
        except Exception as e:
            print(f"Error processing surrogate {i+1}: {e}")
            n_surrogates_errors += 1
            continue
        
        # Print progress every 20 surrogates
        if (i + 1) % 20 == 0:
            print(f"  Processed {i + 1}/{min(500, n_surrogates)} surrogates")

    surrogate_correlations = np.array(surrogate_correlations)
    
    # Calculate original (empirical map) correlation for reference
    invert_flag = map_invert_mapping.get(map_name, True)  # Default to True if map not found
    empirical_model = Bnm(sc, maps=empirical_maps[map_name], map_invert_flags=[invert_flag])
    empirical_model.set('w_EI', (current_theta[0,0], current_theta[1,0]))
    empirical_model.set('w_EE', (current_theta[2,0], current_theta[3,0]))
    empirical_model.set('G', current_theta[4,0])
    
    empirical_corr_bold_none = False
    empirical_error = False
    
    try:
        empirical_model.moments_method()
        
        # Check if corr_bold exists and is not None
        model_FC_empirical = empirical_model.get('corr_bold')
        if model_FC_empirical is None:
            print(f"Warning: corr_bold is None for {map_name} empirical map. Skipping this map type.")
            empirical_corr_bold_none = True
            continue
            
        model_fc_empirical = subdiag(model_FC_empirical)
        original_corr, _ = pearsonr(empirical_fc, model_fc_empirical)
        
    except Exception as e:
        print(f"Error processing {map_name} empirical model: {e}")
        empirical_error = True
        continue
    
    # Calculate proportions
    proportion_surrogates_none = n_surrogates_none / n_surrogates_processed if n_surrogates_processed > 0 else 0
    proportion_surrogates_errors = n_surrogates_errors / n_surrogates_processed if n_surrogates_processed > 0 else 0
    proportion_surrogates_success = len(surrogate_correlations) / n_surrogates_processed if n_surrogates_processed > 0 else 0
    
    # Store results
    all_results[map_name] = {
        'surrogate_correlations': surrogate_correlations,
        'original_correlation': original_corr,
        'n_surrogates': len(surrogate_correlations),  # Actual number processed successfully
        'n_surrogates_processed': n_surrogates_processed,
        'n_surrogates_none': n_surrogates_none,
        'n_surrogates_errors': n_surrogates_errors,
        'proportion_surrogates_none': proportion_surrogates_none,
        'proportion_surrogates_errors': proportion_surrogates_errors,
        'proportion_surrogates_success': proportion_surrogates_success,
        'empirical_corr_bold_none': empirical_corr_bold_none,
        'empirical_error': empirical_error
    }
    
    print(f"\n{map_name.upper()} Results:")
    print(f"  Original correlation: {original_corr:.3f}")
    print(f"  Empirical model corr_bold None: {'Yes' if empirical_corr_bold_none else 'No'}")
    print(f"  Surrogate processing:")
    print(f"    Total processed: {n_surrogates_processed}")
    print(f"    Successful: {len(surrogate_correlations)} ({proportion_surrogates_success:.1%})")
    print(f"    corr_bold None: {n_surrogates_none} ({proportion_surrogates_none:.1%})")
    print(f"    Errors: {n_surrogates_errors} ({proportion_surrogates_errors:.1%})")
    
    if len(surrogate_correlations) > 0:
        print(f"  Surrogate correlations - Mean: {np.mean(surrogate_correlations):.3f}, Std: {np.std(surrogate_correlations):.3f}")
        print(f"  Min: {np.min(surrogate_correlations):.3f}, Max: {np.max(surrogate_correlations):.3f}")
    else:
        print(f"  No valid surrogate correlations computed")

print(f"\n{'='*60}")
print("SUMMARY COMPARISON")
print(f"{'='*60}")
for map_name, results in all_results.items():
    if len(results['surrogate_correlations']) > 0:
        surrogate_mean = np.mean(results['surrogate_correlations'])
        original = results['original_correlation']
        improvement = original - surrogate_mean
        print(f"{map_name.upper():15} | Original: {original:.3f} | Surrogate Mean: {surrogate_mean:.3f} | Improvement: {improvement:.3f}")
    else:
        print(f"{map_name.upper():15} | Original: {results['original_correlation']:.3f} | No valid surrogates")

print(f"\n{'='*60}")
print("corr_bold NONE PROPORTIONS BY SIMULATION")
print(f"{'='*60}")
print(f"{'Map Type':15} | {'Empirical None':13} | {'Surrogate None':15} | {'Surrogate Errors':16} | {'Success Rate':12}")
print("-" * 75)
for map_name, results in all_results.items():
    empirical_status = "Yes" if results['empirical_corr_bold_none'] else "No"
    none_pct = f"{results['proportion_surrogates_none']:.1%}"
    error_pct = f"{results['proportion_surrogates_errors']:.1%}"
    success_pct = f"{results['proportion_surrogates_success']:.1%}"
    print(f"{map_name.upper():15} | {empirical_status:13} | {none_pct:15} | {error_pct:16} | {success_pct:12}")

Processing all combinations of empirical and surrogate maps...

Processing MYELIN empirical map
Loaded 500 surrogate maps for myelin
Using theta values from myelin
  Processed 20/500 surrogates
  Processed 40/500 surrogates
  Processed 60/500 surrogates
  Processed 80/500 surrogates
  Processed 100/500 surrogates
  Processed 120/500 surrogates
  Processed 140/500 surrogates
  Processed 160/500 surrogates
  Processed 180/500 surrogates
  Processed 200/500 surrogates
  Processed 220/500 surrogates
  Processed 240/500 surrogates
  Processed 260/500 surrogates
  Processed 280/500 surrogates
  Processed 300/500 surrogates
  Processed 320/500 surrogates
  Processed 340/500 surrogates
  Processed 360/500 surrogates
  Processed 380/500 surrogates
  Processed 400/500 surrogates
  Processed 420/500 surrogates
  Processed 440/500 surrogates
  Processed 460/500 surrogates
  Processed 480/500 surrogates
  Processed 500/500 surrogates

MYELIN Results:
  Original correlation: 0.517
  Empirical model 

In [ ]:
# Remove empty subplots
for idx in range(len(all_results), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.suptitle('Empirical vs Surrogate Map Correlations', fontsize=16, y=1.02)
plt.show()

# Summary comparison plot
plt.figure(figsize=(12, 8))
map_names = list(all_results.keys())
original_corrs = [all_results[name]['original_correlation'] for name in map_names]
surrogate_means = [np.mean(all_results[name]['surrogate_correlations']) for name in map_names]
surrogate_stds = [np.std(all_results[name]['surrogate_correlations']) for name in map_names]

x_pos = np.arange(len(map_names))
plt.bar(x_pos, original_corrs, alpha=0.7, label='Original Maps', color='red')
plt.errorbar(x_pos, surrogate_means, yerr=surrogate_stds, fmt='o', 
             color='blue', capsize=5, label='Surrogate Maps (mean ± std)')

plt.xlabel('Map Type')
plt.ylabel('Correlation with Empirical FC')
plt.title('Comparison of Original vs Surrogate Map Performance')
plt.xticks(x_pos, [name.capitalize() for name in map_names])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("SUMMARY COMPARISON")
print(f"{'='*60}")
for map_name, results in all_results.items():
    surrogate_mean = np.mean(results['surrogate_correlations'])
    original = results['original_correlation']
    improvement = original - surrogate_mean
    print(f"{map_name.upper():15} | Original: {original:.3f} | Surrogate Mean: {surrogate_mean:.3f} | Improvement: {improvement:.3f}")

## Correlation

In [ ]:
# Calculate Pearson correlation between empirical and model FC
empirical_fc = subdiag(fc_obj)
model_fc = subdiag(heterogeneous.get('corr_bold'))
model_fc_shuffled = subdiag(heterogeneous_shuffled.get('corr_bold'))

# Calculate correlations
corr, p_value = pearsonr(empirical_fc, model_fc)
corr_shuffled, p_value_shuffled = pearsonr(empirical_fc, model_fc_shuffled)

print(f"Pearson correlation (original map): {corr:.3f}, p-value: {p_value:.3e}")
print(f"Pearson correlation (shuffled map): {corr_shuffled:.3f}, p-value: {p_value_shuffled:.3e}")


## Plotting Logic

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def matrix_plot(ax, x, cmap, add_colorbar=True, n_ticks=5):
    im = ax.pcolormesh(x, cmap=cmap, vmin=x.min(), vmax=x.max())
    ax.set_aspect(1)
    ax.set_xlim([0, x.shape[1]])
    ax.set_ylim([0, x.shape[0]])
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    
    if add_colorbar:
        cbar = plt.colorbar(im, ax=ax)
        # Create evenly spaced ticks from min to max
        ticks = np.linspace(x.min(), x.max(), n_ticks)
        cbar.set_ticks(ticks)
        cbar.set_ticklabels([f'{tick:.3f}' for tick in ticks])
    
    return im


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
matrix_plot(axes[0], fc_obj, 'RdBu_r')
axes[0].set_title('Empirical FC')
matrix_plot(axes[1], model_FC, 'RdBu_r')
axes[1].set_title('Heterogeneous Model FC')
matrix_plot(axes[2], model_FC_heterogeneous_shuffled, 'RdBu_r')
axes[2].set_title('Shuffled Model FC')
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 6))
plt.imshow(model_FC, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(label='FC strength')
plt.title('Model Functional Connectivity Matrix')
plt.xlabel('Regions')
plt.ylabel('Regions')
plt.show()